# Mouse Button Remapper

A small standalone replacement for the parts of Mac Mouse Fix you use, written from scratch in Swift with public Apple APIs (plus the reverse-engineered dock-swipe event for the Spaces drag).

**Source:** `remap.swift` in this folder. **Binary:** `remap`.

| Button | Click | Hold | Double click | Double click + hold | Drag |
|---|---|---|---|---|---|
| Middle | middle click | Mission Control | Shift+Return | Look Up | Spaces & Mission Control |
| Button 4 | Back (Cmd+[) | Cmd+V | | | |
| Button 5 | Forward (Cmd+]) | Cmd+C | | | |

The tool needs **Accessibility** permission for whatever app launches it (Jupyter's parent app, Terminal, or VS Code). macOS asks on first run.

## 1. Build

In [ ]:
%cd "/Users/cagri/Desktop/mac mouse fix"
!swiftc -O remap.swift -o remap && ls -la remap

## 2. Stop Mac Mouse Fix first

Its helper intercepts the buttons before this tool can. Do not open the Mac Mouse Fix window while testing: opening it relaunches the helper.

In [ ]:
!osascript -e 'tell application "Mac Mouse Fix" to quit' 2>/dev/null; launchctl bootout gui/$(id -u)/com.nuebling.mac-mouse-fix.helper 2>/dev/null; pkill -f 'Mac Mouse Fix' 2>/dev/null; sleep 1; pgrep -fl 'Mac Mouse Fix' || echo 'Mac Mouse Fix stopped'

## 3. Run the remapper in the background

In [ ]:
import subprocess, time
log = open('/private/tmp/claude-501/-Users-cagri-Desktop-mac-mouse-fix/68dcd2fc-45f8-40d7-b878-9532a9247520/scratchpad/remap.log', 'w')
proc = subprocess.Popen(['./remap'], stdout=log, stderr=subprocess.STDOUT, cwd='/Users/cagri/Desktop/mac mouse fix')
time.sleep(2)
print('running' if proc.poll() is None else f'exited with code {proc.returncode}')
print(open('/private/tmp/claude-501/-Users-cagri-Desktop-mac-mouse-fix/68dcd2fc-45f8-40d7-b878-9532a9247520/scratchpad/remap.log').read())

## 4. Test, then read the log

Press each button and gesture, then run this cell. Every button event the tool receives and every action it sends is listed. If a button prints nothing, the mouse reports it under a different number: run `./remap --learn` from a terminal to see the numbers.

In [ ]:
print(open('/private/tmp/claude-501/-Users-cagri-Desktop-mac-mouse-fix/68dcd2fc-45f8-40d7-b878-9532a9247520/scratchpad/remap.log').read()[-4000:])

## 5. Stop the remapper

In [ ]:
proc.terminate(); print('stopped')

## 6. Find your button numbers (optional)

Runs for 15 seconds and prints the number of each button you press.

In [ ]:
import subprocess
try:
    print(subprocess.run(['./remap','--learn'], cwd='/Users/cagri/Desktop/mac mouse fix', capture_output=True, text=True, timeout=15).stdout)
except subprocess.TimeoutExpired as e:
    print(e.stdout.decode() if e.stdout else '(no buttons pressed)')

## 7. Tuning

Constants near the top of `remap.swift`:

- `holdDelay` (0.35 s): press longer than this counts as Hold
- `doubleClickWindow` (0.3 s): second click within this is a Double Click; also the delay before a single middle click fires
- `dragThreshold` (8 px): movement before a press becomes a Drag
- `originOffsetPerSpace` (1.5): Spaces swipe scale. 2.0 for one Space, 1.5 for three, 1.33 for four

Edit, then rerun the Build cell and restart.

## 8. Start at login (optional)

System Settings > General > Login Items & Extensions > Open at Login > plus button > choose the `remap` binary. Or leave a terminal tab running `./remap`.

## 9. Remove Mac Mouse Fix completely (when you decide to)

Quit it first, then run this cell. It removes the app, its helper registration, preferences and caches, and revokes nothing automatically: afterwards delete it from System Settings > Privacy & Security > Accessibility and Input Monitoring if it is still listed.

In [ ]:
# Uncomment to run. This deletes Mac Mouse Fix and all its files for this user.
# !launchctl bootout gui/$(id -u)/com.nuebling.mac-mouse-fix.helper 2>/dev/null; rm -rf '/Applications/Mac Mouse Fix.app' ~/Library/LaunchAgents/com.nuebling.mac-mouse-fix.helper.plist ~/Library/Application\ Support/com.nuebling.mac-mouse-fix ~/Library/Preferences/com.nuebling.mac-mouse-fix.plist ~/Library/Preferences/com.nuebling.mac-mouse-fix.helper.plist ~/Library/Caches/com.nuebling.mac-mouse-fix* ~/Library/HTTPStorages/com.nuebling.mac-mouse-fix*; echo removed